<a href="https://colab.research.google.com/github/kshelp/pytorchtrf/blob/main/09%EC%9E%A5%20%EA%B0%9D%EC%B2%B4%20%ED%83%90%EC%A7%80/license_plate_yolov8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
######################
# 0. 코랩 환경 준비
from google.colab import drive
drive.mount('/content/drive')

!pip install ultralytics opencv-python easyocr

!wget -O /content/drive/MyDrive/models/license_plate_detector.pt \
https://raw.githubusercontent.com/Muhammad-Zeerak-Khan/Automatic-License-Plate-Recognition-using-YOLOv8/main/license_plate_detector.pt

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
--2026-03-18 00:58:40--  https://raw.githubusercontent.com/Muhammad-Zeerak-Khan/Automatic-License-Plate-Recognition-using-YOLOv8/main/license_plate_detector.pt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.110.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 6241454 (6.0M) [application/octet-stream]
Saving to: ‘/content/drive/MyDrive/models/license_plate_detector.pt’

/content/drive/MyDr 100%[===================>]   5.95M  38.7MB/s    in 0.2s    

2026-03-18 00:58:40 (38.7 MB/s) - ‘/content/drive/MyDrive/models/license_plate_detector.pt’ saved [6241454/6241454]



In [13]:
######################
# 1. 차량번호 식별
import cv2
import numpy as np
import re
from ultralytics import YOLO
import easyocr

# =========================
# 모델 로드
# =========================
# https://github.com/Muhammad-Zeerak-Khan/Automatic-License-Plate-Recognition-using-YOLOv8/blob/main/license_plate_detector.pt
model = YOLO("/content/drive/MyDrive/models/license_plate_detector.pt")
reader = easyocr.Reader(['ko', 'en'], gpu=False)

# =========================
# 이미지 로드
# =========================
img = cv2.imread("/content/drive/MyDrive/datasets/car001.jpg")
img = np.ascontiguousarray(img)

# =========================
# YOLO 추론
# =========================
results = model(img)

# =========================
# 번호판 처리
# =========================
for r in results:
    for box in r.boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0])

        # 번호판 Crop
        plate_img = img[y1:y2, x1:x2]
        if plate_img.size == 0:
            continue

        # =========================
        # OCR 전처리 (한글 번호판 핵심)
        # =========================
        gray = cv2.cvtColor(plate_img, cv2.COLOR_BGR2GRAY)
        gray = cv2.bilateralFilter(gray, 11, 17, 17)
        _, thresh = cv2.threshold(gray, 0, 255,
                                  cv2.THRESH_BINARY + cv2.THRESH_OTSU)

        # =========================
        # OCR
        # =========================
        ocr_result = reader.readtext(
            thresh,
            detail=1,
            paragraph=False,
            text_threshold=0.6,
            low_text=0.3
        )

        # =========================
        # 한글 번호판 후처리
        # =========================
        plate_text = ""
        for (_, text, conf) in ocr_result:
            if conf > 0.4:
                # 숫자 + 한글만 허용
                clean_text = re.sub(r'[^0-9가-힣]', '', text)
                plate_text += clean_text

        # 번호판 형식 보정 (예: 12가3456)
        plate_text = plate_text.strip()

        # =========================
        # 시각화
        # =========================
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(
            img,
            plate_text,
            (x1, y1 - 10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.9,
            (0, 255, 0),
            2
        )

# =========================
# 결과 출력
# =========================
#cv2.imshow("License Plate Recognition (KR)", img)
#cv2.waitKey(0)
#cv2.destroyAllWindows()
print("차량번호="+plate_text)


0: 384x640 1 license_plate, 150.0ms
Speed: 6.0ms preprocess, 150.0ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)
차량번호=45가6492
